# Summit-Sim: Complete E2E Workflow Demo

Demonstrates the full Summit-Sim workflow:
1. **Scenario Generation** - AI generates wilderness rescue scenario
2. **Simulation** - Student navigates scenario with human-in-the-loop
3. **Debrief** - Performance analysis and learning report

All phases are traced to MLflow for visibility.

## 1. Setup & Imports

In [1]:
import warnings

# Suppress autologging warnings
warnings.filterwarnings("ignore", message=".*autologging.*")
warnings.filterwarnings("ignore", message=".*LangChain.*")

from summit_sim.agents.debrief import generate_debrief
from summit_sim.agents.generator import generate_scenario
from summit_sim.graphs.simulation import create_simulation_graph
from summit_sim.schemas import HostConfig, generate_scenario_id
from summit_sim.settings import settings
from summit_sim.tracing import enable_tracing, simulation_session

# Initialize MLflow tracing
enable_tracing()

print(f"✓ MLflow: {settings.mlflow_tracking_uri}")
print(f"✓ Experiment: {settings.mlflow_experiment_name}")
print(f"✓ Model: {settings.default_model}")
print(f"✓ API Key: {bool(settings.openrouter_api_key)}")

✓ MLflow: https://mlflow.bhamm-lab.com
✓ Experiment: summit-sim
✓ Model: google/gemini-3.1-flash-lite-preview
✓ API Key: True


## 2. Phase 1: Scenario Generation

Generate a complete wilderness rescue scenario from minimal host configuration.

In [2]:
# Configuration
config = HostConfig(
    num_participants=3,
    activity_type="hiking",
    difficulty="med",
    class_id="demo-class-2024",
)

print(
    f"Generating: {config.activity_type} scenario for {config.num_participants} participants"
)
print("-" * 60)

# Generate scenario
scenario = await generate_scenario(config)
scenario_id = generate_scenario_id()

print(f"✓ Title: {scenario.title}")
print(f"✓ Setting: {scenario.setting}")
print(f"✓ Patient: {scenario.patient_summary}")
print(f"✓ Turns: {len(scenario.turns)}")
print(f"✓ Scenario ID: {scenario_id}")
print("\nLearning Objectives:")
for obj in scenario.learning_objectives:
    print(f"  • {obj}")

Generating: hiking scenario for 3 participants
------------------------------------------------------------
✓ Title: The Crestline Cough
✓ Setting: The "Crestline Trail," an exposed, high-altitude alpine hiking path. It is mid-afternoon, 38°F, and windy. The team is 4 miles from the trailhead.
✓ Patient: 35-year-old female, physically fit, hiker. Complaining of severe "tightness in chest," significant shortness of breath at rest, and a non-productive cough that has developed over the last two hours. Height of current location: 11,000 feet.
✓ Turns: 4
✓ Scenario ID: scn-6cb74764

Learning Objectives:
  • Recognize symptoms of severe altitude-related illness (HAPE) over minor altitude sickness.
  • Prioritize immediate descent as the definitive treatment for HAPE.
  • Perform focused medical assessment in a wilderness setting.


Trace(trace_id=tr-97a343032c156f5455f2d2523729f2e3)

### Display Scenario Structure

In [3]:
for turn in scenario.turns:
    marker = "(START)" if turn.is_starting_turn else ""
    print(f"\n{'=' * 60}")
    print(f"Turn {turn.turn_id} {marker}")
    print(f"{'=' * 60}")
    print(
        turn.narrative_text[:200] + "..."
        if len(turn.narrative_text) > 200
        else turn.narrative_text
    )
    print("\nChoices:")
    for choice in turn.choices:
        mark = "✓" if choice.is_correct else " "
        next_turn = (
            "END" if choice.next_turn_id is None else f"Turn {choice.next_turn_id}"
        )
        print(f"  [{mark}] {choice.description[:60]}... → {next_turn}")


Turn 0 (START)
You are hiking at 11,000 feet when one of your group, Jena (35F), slows down significantly. She looks pale and is complaining of a 'tightening' in her chest, severe fatigue, and is starting to cough. ...

Choices:
  [✓] Perform a thorough patient assessment (ABCDE) and check for ... → Turn 1
  [ ] Assume it is just altitude sickness, have her eat a snack, d... → Turn 2

Turn 1 
Your assessment reveals: Pulse 120 bpm, Respirations 28 bpm. Auscultation (listening to lungs) shows distinct crackles at the base of the right lung. She is struggling to speak in full sentences. What...

Choices:
  [✓] Acknowledge the severity and immediately begin descent to lo... → Turn 3
  [ ] Advise her to try and continue slowly hiking up to the plann... → Turn 3

Turn 2 
After an hour, Jena is not improved. Her breathing is more labored, she is agitated, and she is now coughing up thin, frothy-looking mucus. How do you proceed?

Choices:
  [✓] The patient is feeling worse; now observe a bl

## 3. Phase 2: Simulation

Run the interactive simulation with human-in-the-loop.
The graph will pause at each turn for you to make a choice.

In [4]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

# Create graph with memory
memory = InMemorySaver()
graph = create_simulation_graph(checkpointer=memory)

# Initialize state
initial_state = {
    "scenario_draft": scenario,
    "current_turn_id": scenario.starting_turn_id,
    "transcript": [],
    "is_complete": False,
    "key_learning_moments": [],
    "last_selected_choice": None,
    "simulation_result": None,
    "scenario_id": scenario_id,
    "class_id": config.class_id,
}

print("✓ Graph initialized")
print(f"✓ Starting at turn {initial_state['current_turn_id']}")

✓ Graph initialized
✓ Starting at turn 0


In [5]:
# Run simulation with MLflow session tracking
from summit_sim.tracing import log_simulation_results

with simulation_session(config, scenario_id=scenario_id) as (session_id, graph_config):
    print(f"\n📋 Session ID: {session_id}")
    print("🎮 Starting simulation...\n")
    print("=" * 60)
    print("INTERACTIVE SIMULATION")
    print("=" * 60 + "\n")

    current_state = initial_state
    turn_count = 0
    simulation_complete = False
    first_turn = True

    while not simulation_complete:
        turn_count += 1

        # Get the current turn from the graph state
        current_turn_id = current_state["current_turn_id"]
        current_turn = scenario.get_turn(current_turn_id)

        if current_turn is None:
            print(f"Error: Turn {current_turn_id} not found")
            break

        # Display turn
        print(f"\n--- TURN {turn_count} ---\n")
        print(current_turn.narrative_text)
        print("\nChoices:")
        for i, choice in enumerate(current_turn.choices, 1):
            print(f"  {i}. {choice.description}")

        # Get user input
        while True:
            try:
                sel = int(input("\nSelect choice (number): ")) - 1
                if 0 <= sel < len(current_turn.choices):
                    break
                print("Invalid selection")
            except ValueError:
                print("Please enter a number")

        selected_choice = current_turn.choices[sel]

        # Process the turn using ainvoke to maintain tracing context
        if first_turn:
            # First turn: pass full state to initialize the graph
            current_state = await graph.ainvoke(
                initial_state,
                graph_config,
            )
            first_turn = False
        else:
            # Subsequent turns: use Command to resume
            current_state = await graph.ainvoke(
                Command(resume={"choice_id": selected_choice.choice_id}),
                graph_config,
            )

        # Check if complete
        if current_state.get("is_complete"):
            simulation_complete = True

        # Safety limit
        if turn_count > 10:
            print("\nSafety limit reached")
            break

    print("\n" + "=" * 60)
    print("SIMULATION COMPLETE")
    print("=" * 60)

    # Log results
    log_simulation_results(
        transcript=current_state["transcript"],
        is_complete=current_state["is_complete"],
        key_learning_moments=current_state["key_learning_moments"],
    )
    print("\n✓ Results logged to MLflow")
    print(f"✓ Total turns: {len(current_state['transcript'])}")
    print(f"✓ Learning moments: {len(current_state['key_learning_moments'])}")


📋 Session ID: f2d5eaae-ed3f-4155-ac78-6fde7e971caa
🎮 Starting simulation...

INTERACTIVE SIMULATION


--- TURN 1 ---

You are hiking at 11,000 feet when one of your group, Jena (35F), slows down significantly. She looks pale and is complaining of a 'tightening' in her chest, severe fatigue, and is starting to cough. You are 4 miles from the trailhead and the weather is cooling quickly. What is your immediate action?

Choices:
  1. Perform a thorough patient assessment (ABCDE) and check for specific HAPE symptoms.
  2. Assume it is just altitude sickness, have her eat a snack, drink water, and rest for an hour.


2026/03/23 21:54:12 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f5a222cf2e0> at 0x7f59e6c43b40> was created in a different Context
2026/03/23 21:54:12 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f5a222cf2e0> at 0x7f59e6c4ee00> was created in a different Context
2026/03/23 21:54:12 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f5a222cf2e0> at 0x7f59e75c52c0> was created in a different Context



--- TURN 2 ---

You are hiking at 11,000 feet when one of your group, Jena (35F), slows down significantly. She looks pale and is complaining of a 'tightening' in her chest, severe fatigue, and is starting to cough. You are 4 miles from the trailhead and the weather is cooling quickly. What is your immediate action?

Choices:
  1. Perform a thorough patient assessment (ABCDE) and check for specific HAPE symptoms.
  2. Assume it is just altitude sickness, have her eat a snack, drink water, and rest for an hour.


2026/03/23 21:54:15 WARNING mlflow.tracing.fluent: Failed to start span LangGraph: 'NoneType' object has no attribute 'set_span_type'. For full traceback, set logging level to debug.
Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
2026/03/23 21:54:15 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f5a222cf2e0> at 0x7f59e6bdad80> was created in a different Context
2026/03/23 21:54:20 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f5a222cf2e0> at 0x7f59e6bdabc0> was created in a different Context
2026/03/23 21:54:20 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVa


--- TURN 3 ---

Your assessment reveals: Pulse 120 bpm, Respirations 28 bpm. Auscultation (listening to lungs) shows distinct crackles at the base of the right lung. She is struggling to speak in full sentences. What do you do?

Choices:
  1. Acknowledge the severity and immediately begin descent to lower altitude while keeping her warm.
  2. Advise her to try and continue slowly hiking up to the planned camp to rest.


Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
Deserializing unregistered type summit_sim.schemas.ChoiceOption from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ChoiceOption')]
Deserializing unregistered type summit_sim.schemas.SimulationResult from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'SimulationResult')]
2026/03/23 21:54:32 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f5a222cf2e0> at 0x7f59e6aba580> was created in a different Context
2026/03/23 21:54:40 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVa


--- TURN 4 ---

The outcome of your decisions is revealed. Based on your actions, either immediate care or delayed intervention has significantly impacted the patient's prognosis.

Choices:
  1. Scenario Complete: You recognized the severity of HAPE and prioritized descent. The patient's condition stabilizes as you lower altitude.
  2. Scenario Complete: You delayed definitive treatment (descent). The patient's condition worsened significantly, requiring an emergency helicopter evacuation due to pulmonary collapse.


2026/03/23 21:54:47 WARNING mlflow.tracing.fluent: Failed to start span LangGraph: 'NoneType' object has no attribute 'set_span_type'. For full traceback, set logging level to debug.
Deserializing unregistered type summit_sim.schemas.ScenarioDraft from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ScenarioDraft')]
Deserializing unregistered type summit_sim.schemas.ChoiceOption from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'ChoiceOption')]
Deserializing unregistered type summit_sim.schemas.SimulationResult from checkpoint. This will be blocked in a future version. Add to allowed_msgpack_modules to silence: [('summit_sim.schemas', 'SimulationResult')]
2026/03/23 21:54:47 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during autologging: <Token var=<ContextVar name='current_context' default={} at 0x7f5a222cf2e0> a


SIMULATION COMPLETE

✓ Results logged to MLflow
✓ Total turns: 3
✓ Learning moments: 6
🏃 View run sim-demo-class-2024-hiking-3p-med at: https://mlflow.bhamm-lab.com/#/experiments/7/runs/5044723512ca4c82afb0bda837d4f11a
🧪 View experiment at: https://mlflow.bhamm-lab.com/#/experiments/7


[Trace(trace_id=tr-20e1230f78b8699e6a434dc325cf7b36), Trace(trace_id=tr-d6979828e4282bdc887020bb49fa3734), Trace(trace_id=tr-f5cc93c59814ab85e84f3fe400d53c92), Trace(trace_id=tr-af102a0adaf25403bcfe80523473cd0a)]

## 4. Phase 3: Debrief

Generate comprehensive performance report from simulation transcript.

In [6]:
print("\nGenerating debrief report...\n")

debrief_report = await generate_debrief(
    transcript=current_state["transcript"],
    scenario_draft=scenario,
    scenario_id=scenario_id,
)

print("=" * 60)
print("DEBRIEF REPORT")
print("=" * 60)

print(f"\n📊 Final Score: {debrief_report.final_score:.1f}%")
status_emoji = "✅" if debrief_report.completion_status == "pass" else "❌"
print(f"{status_emoji} Status: {debrief_report.completion_status.upper()}")

print("\n📝 Summary:")
print(debrief_report.summary)

if debrief_report.key_mistakes:
    print("\n⚠️ Key Mistakes:")
    for mistake in debrief_report.key_mistakes:
        print(f"  • {mistake}")

if debrief_report.strong_actions:
    print("\n💪 Strong Actions:")
    for action in debrief_report.strong_actions:
        print(f"  • {action}")

if debrief_report.teaching_points:
    print("\n📚 Teaching Points:")
    for point in debrief_report.teaching_points:
        print(f"  • {point}")

if debrief_report.best_next_actions:
    print("\n🎯 Recommendations:")
    for rec in debrief_report.best_next_actions:
        print(f"  • {rec}")


Generating debrief report...

DEBRIEF REPORT

📊 Final Score: 66.7%
❌ Status: FAIL

📝 Summary:
The student successfully initiated a structured assessment approach, demonstrating good initial clinical judgment. However, the simulation hit a critical juncture where the student initially misjudged the severity of HAPE symptoms, recommending rest at altitude. This is a common but dangerous pitfall in mountain medicine. The student corrected the course in the subsequent turn, recognizing that immediate descent is the non-negotiable definitive treatment. Overall, the student demonstrates the necessary assessment skills but needs to strengthen their decision-making speed regarding emergency evacuations at altitude.

⚠️ Key Mistakes:
  • Recommended rest at current altitude instead of immediate descent for a patient exhibiting symptoms of High Altitude Pulmonary Edema (HAPE).
  • Underestimated the rapid, life-threatening progression of pulmonary edema at high altitude.

💪 Strong Actions:
  • 

Trace(trace_id=tr-f32c0d6b2b5ea2f19c832fa9e658e9ec)

## 5. Results Summary

Complete transcript and learning moments from the simulation.

In [7]:
print("\n" + "=" * 60)
print("COMPLETE TRANSCRIPT")
print("=" * 60 + "\n")

for i, entry in enumerate(current_state["transcript"], 1):
    status = "✓" if entry["was_correct"] else "✗"
    print(f"Turn {i} {status}")
    print(f"  Situation: {entry['turn_narrative'][:80]}...")
    print(f"  Choice: {entry['choice_description']}")
    print(f"  Feedback: {entry['feedback'][:100]}...")
    print()


COMPLETE TRANSCRIPT

Turn 1 ✓
  Situation: You are hiking at 11,000 feet when one of your group, Jena (35F), slows down sig...
  Choice: Perform a thorough patient assessment (ABCDE) and check for specific HAPE symptoms.
  Feedback: Excellent choice! Jumping to assumptions can be dangerous at altitude; performing a systematic asses...

Turn 2 ✗
  Situation: Your assessment reveals: Pulse 120 bpm, Respirations 28 bpm. Auscultation (liste...
  Choice: Advise her to try and continue slowly hiking up to the planned camp to rest.
  Feedback: While resting might seem logical for general fatigue, your patient is exhibiting life-threatening sy...

Turn 3 ✓
  Situation: The outcome of your decisions is revealed. Based on your actions, either immedia...
  Choice: Scenario Complete: You recognized the severity of HAPE and prioritized descent. The patient's condition stabilizes as you lower altitude.
  Feedback: Excellent work identifying the signs of High Altitude Pulmonary Edema (HAPE). Your de

In [8]:
print("\n" + "=" * 60)
print("KEY LEARNING MOMENTS")
print("=" * 60 + "\n")

for i, moment in enumerate(current_state["key_learning_moments"], 1):
    print(f"{i}. {moment}")


KEY LEARNING MOMENTS

1. Systematic assessment (ABCDE) is the gold standard for prioritizing care in wilderness settings, especially when symptoms are non-specific.
2. Always treat persistent cough and chest tightness at altitude as HAPE until proven otherwise, as it can deteriorate rapidly.
3. Crackles during auscultation at altitude are a definitive 'red flag' for HAPE and supersede common 'rest and recover' strategies.
4. When faced with signs of pulmonary edema (HAPE) or high-altitude cerebral edema (HACE), the priority must always be immediate descent; delaying for any reason significantly increases the risk of a fatal outcome.
5. Severe dyspnea at rest and a persistent, non-productive cough at high altitudes are critical markers for HAPE, which must be treated as a medical emergency rather than mild altitude illness.
6. Descent is the primary and non-negotiable intervention for HAPE; any delay in losing altitude significantly increases the risk of rapid, life-threatening respira

---
## ✅ Complete Workflow Demo

All phases executed:
- ✓ Scenario generated with AI
- ✓ Simulation completed with human-in-the-loop
- ✓ Debrief report generated
- ✓ All traces logged to MLflow

View traces in MLflow UI at your configured tracking URI.